# Notebook 13: Phase 2G Error Analysis & False Alert Suppression
## Evaluation of Failure Modes: Speech vs Yawns, Glasses, Lighting, Head Rotation, and Face Loss

### Objective
Inspect and quantify the edge cases identified during Phase 1 repository audit and Phase 2B/2F audits:
1. Conversational speech / laughing vs physiological yawns
2. Normal extended blinks (300-400ms) vs micro-sleeps (>800ms)
3. Eyeglasses, glare, and shadow occlusions
4. Head pose rotations (mirror checks) vs drowsiness nodding
5. Sudden face loss and graceful timeout handling


In [1]:
import sys
from pathlib import Path
import csv, json
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.drowsiness_engine import MultiSignalFusionEngine
from src.physiological import TemporalSignal

print('Loaded Error Analysis & Edge Case Engine.')


Loaded Error Analysis & Edge Case Engine.


### 2. Edge Case Benchmark Results
Comparison of legacy frame-counter behavior vs Phase 2G multi-signal fusion behavior.


In [2]:
with open(PROJECT_ROOT / 'results/phase2g/benchmark_summary.json', 'r') as f:
    summary = json.load(f)

cases = summary['error_analysis']
for i, c in enumerate(cases, 1):
    print(f"[{i}] Condition: {c['condition']}")
    print(f"    Description:  {c['description']}")
    print(f"    Legacy:       {c['legacy_behavior']}")
    print(f"    Phase 2G:     {c['phase2g_behavior']}")
    print(f"    Reduction:    {c['false_alarm_reduction']}\n")


[1] Condition: Conversational Speech / Talking
    Description:  Rapid mouth opening and closing (<1.5s duration)
    Legacy:       False Warning Alert (yawn counter > 15 frames)
    Phase 2G:     Suppressed (Speech duration < 2.0s rejected)
    Reduction:    100%

[2] Condition: Normal Extended Blink (300-400ms)
    Description:  Natural slow blink during dry eye or fatigue onset
    Legacy:       Risks Critical Alert (near 700ms fixed threshold)
    Phase 2G:     Classified as NORMAL_BLINK; eye score capped < 35
    Reduction:    95%

[3] Condition: Eyeglasses & Frame Shadow
    Description:  Thick spectacle frames partially obscuring eye landmarks
    Legacy:       YOLO visual detector handles if trained; no consensus
    Phase 2G:     Dual consensus (YOLO conf + EAR fallback certainty)
    Reduction:    85%

[4] Condition: Head Turn Off-Road (Checking Mirrors)
    Description:  Driver turns head left/right for 0.5 - 1.2s
    Legacy:       Partial face loss or false closed eye detec

### 3. Speech vs Yawn Differentiation Simulation
Demonstrating mouth opening duration filter (<1.5s speech suppressed, >=2.0s confirmed yawn).


In [3]:
engine = MultiSignalFusionEngine(fps=30.0)

# Simulate speech event: mouth open for 1.0s (30 frames)
speech_scores = []
for i in range(30):
    sig = TemporalSignal(
        timestamp=i/30.0, frame_index=i, fps=30.0, face_detected=True,
        average_ear=0.30, eye_closed=False, mar=0.62,
        yawn_candidate=True, yawn_duration=i/30.0, perclos=0.04,
    )
    alert, bd = engine.update(sig, False, True, True, 0.85, i/30.0, i)
    speech_scores.append(bd.yawn_score)

engine.reset()
# Simulate true physiological yawn: mouth open for 3.5s (105 frames)
yawn_scores = []
for i in range(105):
    dur = i / 30.0
    sig = TemporalSignal(
        timestamp=dur, frame_index=i, fps=30.0, face_detected=True,
        average_ear=0.30, eye_closed=False, mar=0.68,
        yawn_candidate=True, yawn_duration=dur, yawn_event='YAWN_CONFIRMED' if dur >= 2.0 else None, perclos=0.04,
    )
    alert, bd = engine.update(sig, False, True, True, 0.92, dur, i)
    yawn_scores.append(bd.yawn_score)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(30)/30.0, speech_scores, color='#3498db', lw=2, label='Conversational Speech (1.0s) -> Suppressed')
plt.plot(np.arange(105)/30.0, yawn_scores, color='#e74c3c', lw=2, label='True Physiological Yawn (3.5s) -> Confirmed')
plt.axvline(2.0, color='black', ls='--', label='Yawn Confirmation Threshold (2.0s)')
plt.ylabel('Yawn Component Score')
plt.xlabel('Mouth Opening Duration (seconds)')
plt.title('Speech Suppression vs True Yawn Verification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4. Face Loss Timeout & Grace Period Simulation
Demonstrating that sudden face disappearance does not trigger false alerts.


In [4]:
engine = MultiSignalFusionEngine(fps=30.0, face_loss_timeout_seconds=2.0)
states = []
for i in range(90):  # 3.0 seconds total
    alert, bd = engine.update(None, False, False, False, 0.0, i/30.0, i)
    states.append(bd.driver_state)

print('Frames 0-30 (0.0 - 1.0s):', set(states[:30]))
print('Frames 30-60 (1.0 - 2.0s):', set(states[30:60]))
print('Frames 60-90 (2.0 - 3.0s, Post-Timeout):', set(states[60:]))
print('Alerts Triggered During Face Loss:', sum(engine.total_alerts.values()))


Frames 0-30 (0.0 - 1.0s): {'NORMAL'}
Frames 30-60 (1.0 - 2.0s): {'NORMAL'}
Frames 60-90 (2.0 - 3.0s, Post-Timeout): {'FACE_LOST'}
Alerts Triggered During Face Loss: 0


### 5. Key Error Analysis Takeaways
1. **Conversational Speech**: Completely decoupled from yawning via the 2.0s sustained threshold, eliminating the largest single source of false alarms in Phase 1 & 2A.
2. **Natural Blinking**: Normal blinks (<400ms) do not accumulate eye closure frames, eliminating false micro-sleep alerts.
3. **Head Pose Decoupling**: Head turns (mirror checks) enter the `FACE_LOST` grace period without incrementing fatigue scores, while sustained downward pitch (>15 deg for >1.0s) reliably flags microsleep head nodding.
4. **Fail-Safe Integrity**: When landmark confidence drops, the pipeline smoothly relies on YOLO visual detector confidence with appropriate certainty damping.
